In [0]:
%sql
CREATE OR REPLACE TEMP VIEW v_partidos_unificados AS
SELECT Div AS liga, Div AS torneo,
       regexp_extract(_source_file, '_(\\d{4})\\.csv$', 1) AS temporada_txt,
       Date AS fecha_txt, Time AS hora, HomeTeam AS local, AwayTeam AS visitante,
       FTHG AS goles_local_txt, FTAG AS goles_visitante_txt, FTR AS resultado,
       PSCH, PSCD, PSCA, AvgCH, AvgCD, AvgCA, B365CH, B365CD, B365CA, _source_file
FROM workspace.tp_futbol.bronze_partidos_europa
UNION ALL
SELECT 'ARG', League, Season, Date, Time, Home, Away, HG, AG, Res,
       PSCH, PSCD, PSCA, AvgCH, AvgCD, AvgCA, B365CH, B365CD, B365CA, _source_file
FROM workspace.tp_futbol.bronze_partidos_arg;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW v_partidos_tipados AS
SELECT
  liga,
  trim(torneo) AS torneo,
  CASE
    WHEN liga <> 'ARG' THEN concat('20', substr(temporada_txt,1,2), '/', substr(temporada_txt,3,2))
    WHEN temporada_txt RLIKE '^\\d{4}/\\d{4}$' THEN concat(substr(temporada_txt,1,5), substr(temporada_txt,8,2))
    ELSE temporada_txt
  END AS temporada,
  CAST(CASE WHEN length(fecha_txt) = 8 THEN try_to_timestamp(fecha_txt, 'dd/MM/yy')
            ELSE try_to_timestamp(fecha_txt, 'dd/MM/yyyy') END AS DATE) AS fecha,
  hora,
  trim(local) AS local,
  trim(visitante) AS visitante,
  try_cast(goles_local_txt AS INT)     AS goles_local,
  try_cast(goles_visitante_txt AS INT) AS goles_visitante,
  resultado,
  try_cast(PSCH AS DOUBLE)   AS psc_local,   try_cast(PSCD AS DOUBLE)   AS psc_empate,   try_cast(PSCA AS DOUBLE)   AS psc_visitante,
  try_cast(AvgCH AS DOUBLE)  AS avgc_local,  try_cast(AvgCD AS DOUBLE)  AS avgc_empate,  try_cast(AvgCA AS DOUBLE)  AS avgc_visitante,
  try_cast(B365CH AS DOUBLE) AS b365c_local, try_cast(B365CD AS DOUBLE) AS b365c_empate, try_cast(B365CA AS DOUBLE) AS b365c_visitante,
  _source_file
FROM v_partidos_unificados;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW v_partidos_validados AS
SELECT *,
  CASE
    WHEN fecha IS NULL THEN 'fecha_invalida'
    WHEN goles_local IS NULL OR goles_visitante IS NULL
         OR goles_local < 0 OR goles_visitante < 0 THEN 'goles_invalidos'
    WHEN resultado <> CASE WHEN goles_local > goles_visitante THEN 'H'
                           WHEN goles_local < goles_visitante THEN 'A' ELSE 'D' END THEN 'resultado_inconsistente'
    WHEN least(psc_local, psc_empate, psc_visitante, avgc_local, avgc_empate, avgc_visitante) <= 1 THEN 'cuota_imposible'
    WHEN (1/psc_local + 1/psc_empate + 1/psc_visitante - 1) NOT BETWEEN 0 AND 0.25 THEN 'margen_imposible_pinnacle'
    WHEN (1/avgc_local + 1/avgc_empate + 1/avgc_visitante - 1) NOT BETWEEN 0 AND 0.25 THEN 'margen_imposible_promedio'
    WHEN row_number() OVER (PARTITION BY liga, fecha, local, visitante ORDER BY _source_file) > 1 THEN 'duplicado'
  END AS motivo_rechazo
FROM v_partidos_tipados;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.tp_futbol.silver_partidos AS
SELECT sha2(concat_ws('|', liga, fecha, local, visitante), 256) AS partido_id,
       * EXCEPT (motivo_rechazo)
FROM v_partidos_validados
WHERE motivo_rechazo IS NULL;

CREATE OR REPLACE TABLE workspace.tp_futbol.silver_partidos_cuarentena AS
SELECT * FROM v_partidos_validados
WHERE motivo_rechazo IS NOT NULL;

In [0]:
%sql
SELECT 'silver' AS tabla, count(*) AS filas, NULL AS motivo FROM workspace.tp_futbol.silver_partidos
UNION ALL
SELECT 'cuarentena', count(*), motivo_rechazo FROM workspace.tp_futbol.silver_partidos_cuarentena GROUP BY motivo_rechazo;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW v_estadios_wd AS
SELECT * FROM (
  SELECT
    regexp_extract(club, 'Q\\d+$', 0) AS club_qid,
    clubLabel     AS club_nombre,
    estadioLabel  AS estadio,
    try_cast(regexp_extract(coord, 'Point\\(([-0-9.]+) ([-0-9.]+)\\)', 2) AS DOUBLE) AS latitud,
    try_cast(regexp_extract(coord, 'Point\\(([-0-9.]+) ([-0-9.]+)\\)', 1) AS DOUBLE) AS longitud,
    try_cast(capacidad AS INT) AS capacidad
  FROM workspace.tp_futbol.bronze_estadios_wikidata
)
QUALIFY row_number() OVER (PARTITION BY club_qid
                           ORDER BY latitud IS NULL, capacidad DESC NULLS LAST, estadio) = 1;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW v_equipos AS
SELECT
  e.liga,
  e.nombre_footballdata                                   AS equipo,
  nullif(e.club_qid, '')                                  AS club_qid,
  coalesce(w.club_nombre, e.nombre_footballdata)          AS club_nombre,
  coalesce(w.estadio, m.estadio)                          AS estadio,
  coalesce(w.latitud,  try_cast(m.latitud  AS DOUBLE))    AS latitud,
  coalesce(w.longitud, try_cast(m.longitud AS DOUBLE))    AS longitud,
  coalesce(w.capacidad, try_cast(m.capacidad AS INT))     AS capacidad,
  CASE WHEN w.latitud IS NOT NULL THEN 'wikidata' ELSE 'manual' END AS fuente_coordenadas,
  coalesce(m.`precision`, 'estadio')                      AS precision_coordenadas
FROM workspace.tp_futbol.bronze_equipos_equivalencia e
LEFT JOIN v_estadios_wd w ON w.club_qid = e.club_qid
LEFT JOIN workspace.tp_futbol.bronze_estadios_manual m ON m.nombre_footballdata = e.nombre_footballdata;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW v_equipos_validados AS
SELECT *,
  CASE
    WHEN latitud IS NULL OR longitud IS NULL THEN 'sin_coordenadas'
    WHEN liga = 'ARG'  AND NOT (latitud BETWEEN -56 AND -21 AND longitud BETWEEN -74 AND -53) THEN 'coordenadas_fuera_de_region'
    WHEN liga <> 'ARG' AND NOT (latitud BETWEEN  27 AND  61 AND longitud BETWEEN -19 AND  20) THEN 'coordenadas_fuera_de_region'
    WHEN row_number() OVER (PARTITION BY liga, equipo ORDER BY club_qid) > 1 THEN 'duplicado'
  END AS motivo_rechazo
FROM v_equipos;

CREATE OR REPLACE TABLE workspace.tp_futbol.silver_equipos AS
SELECT * EXCEPT (motivo_rechazo) FROM v_equipos_validados WHERE motivo_rechazo IS NULL;

CREATE OR REPLACE TABLE workspace.tp_futbol.silver_equipos_cuarentena AS
SELECT * FROM v_equipos_validados WHERE motivo_rechazo IS NOT NULL;

In [0]:
%sql
SELECT 'silver_equipos' AS control, count(*) AS valor FROM workspace.tp_futbol.silver_equipos
UNION ALL
SELECT 'cuarentena', count(*) FROM workspace.tp_futbol.silver_equipos_cuarentena
UNION ALL
SELECT 'equipos_de_partidos_sin_estadio', count(*) FROM (
  SELECT liga, local AS equipo FROM workspace.tp_futbol.silver_partidos
  UNION
  SELECT liga, visitante FROM workspace.tp_futbol.silver_partidos
) p
LEFT ANTI JOIN workspace.tp_futbol.silver_equipos e USING (liga, equipo);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW v_restricciones AS
SELECT
  trim(liga)                    AS liga,
  try_to_date(desde, 'yyyy-MM-dd') AS desde,
  try_to_date(hasta, 'yyyy-MM-dd') AS hasta,
  lower(trim(regimen))          AS regimen,
  lower(trim(visitantes))       AS visitantes,
  detalle,
  lower(trim(confianza))        AS confianza,
  fuente
FROM workspace.tp_futbol.bronze_restricciones_publico;

CREATE OR REPLACE TEMP VIEW v_restricciones_validadas AS
SELECT *,
  CASE
    WHEN desde IS NULL OR hasta IS NULL THEN 'fecha_invalida'
    WHEN desde > hasta THEN 'rango_invertido'
    WHEN regimen NOT IN ('pleno', 'parcial', 'sin_publico') THEN 'regimen_invalido'
    WHEN visitantes NOT IN ('si', 'no', 'na') THEN 'visitantes_invalido'
    WHEN confianza NOT IN ('alta', 'media', 'baja') THEN 'confianza_invalida'
    WHEN desde <= lag(hasta) OVER (PARTITION BY liga ORDER BY desde) THEN 'solapamiento'
    WHEN desde > date_add(lag(hasta) OVER (PARTITION BY liga ORDER BY desde), 1) THEN 'hueco'
  END AS motivo_rechazo
FROM v_restricciones;

CREATE OR REPLACE TABLE workspace.tp_futbol.silver_restricciones AS
SELECT * EXCEPT (motivo_rechazo) FROM v_restricciones_validadas WHERE motivo_rechazo IS NULL;

CREATE OR REPLACE TABLE workspace.tp_futbol.silver_restricciones_cuarentena AS
SELECT * FROM v_restricciones_validadas WHERE motivo_rechazo IS NOT NULL;

In [0]:
%sql
SELECT 'silver_restricciones' AS control, count(*) AS valor FROM workspace.tp_futbol.silver_restricciones
UNION ALL
SELECT 'cuarentena', count(*) FROM workspace.tp_futbol.silver_restricciones_cuarentena
UNION ALL
SELECT 'partidos_sin_periodo', count(*)
FROM workspace.tp_futbol.silver_partidos p
LEFT ANTI JOIN workspace.tp_futbol.silver_restricciones r
  ON p.liga = r.liga AND p.fecha BETWEEN r.desde AND r.hasta
UNION ALL
SELECT 'partidos_con_mas_de_un_periodo', count(*) FROM (
  SELECT p.partido_id
  FROM workspace.tp_futbol.silver_partidos p
  JOIN workspace.tp_futbol.silver_restricciones r
    ON p.liga = r.liga AND p.fecha BETWEEN r.desde AND r.hasta
  GROUP BY p.partido_id HAVING count(*) > 1
);